[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-linear-reg.ipynb)

# Linear Regression

*AIBits Academy · Machine Learning End To End · Supervised Learning*

The cornerstone of predictive modelling — fitting a straight line through data to forecast continuous outcomes with mathematical precision.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

> **🎯 Intuition First**
>
> Picture a cloud of dots — fabric sold on one axis, revenue on the other. Linear regression draws the **one straight line that sits as close as possible to every dot at once**. Prediction is then just reading off it: hand it a new x, it gives back the y on the line. Everything below — cost function, gradient, normal equation — is simply *how the machine finds that best line* so you don't have to eyeball it.

## The Core Idea

Linear regression assumes a **linear relationship** between one or more input features X and a continuous output y. Given n training samples, we want to find parameters θ (weights) such that ŷ ≈ y.

**Simple linear regression (one feature):** 
 ŷ = θ₀ + θ₁x · where θ₀ = intercept (bias), θ₁ = slope (weight) 
 
**Multiple linear regression (p features):** 
 ŷ = θ₀ + θ₁x₁ + θ₂x₂ + … + θₚxₚ = **X**θ (matrix form)

## Cost Function — Mean Squared Error

We minimise the average squared difference between predictions and true values:

$$J(\theta) = \frac{1}{2m}\sum_i (\hat{y}_i-y_i)^2 = \frac{1}{2m}\lVert \mathbf{X}\theta - \mathbf{y}\rVert^2$$

The factor ½ is a convenience — it cancels the 2 that appears when differentiating.

## Deriving the Gradient

Differentiating J(θ) with respect to θⱼ (chain rule through the squared residual):

$$\frac{\partial J}{\partial \theta_j} = \frac{1}{m}\sum_i (\hat{y}_i-y_i)\cdot x_{ij} \quad\Longrightarrow\quad \nabla J(\theta) = \frac{1}{m}\mathbf{X}^{\top}(\mathbf{X}\theta-\mathbf{y})$$

> **📊 Prerequisite refresher**
>
> Partial derivatives and the chain rule are exactly what's needed to follow this derivation line by line — see the **Calculus for ML** prerequisite page for the full first-principles walkthrough (derivatives, the gradient, and why "step opposite the gradient" is the core mechanism behind every optimiser in this course).

This is exactly the gradient the animation below uses at every step. Because J(θ) is a convex quadratic bowl (no local minima), gradient descent with a suitable learning rate is guaranteed to converge to the global optimum.

## The Normal Equation — A Closed-Form Alternative

For linear regression specifically, we don't need to iterate at all. Setting ∇J(θ) = 0 and solving directly:

$$\mathbf{X}^{\top}\mathbf{X}\theta = \mathbf{X}^{\top}\mathbf{y} \quad\Longrightarrow\quad \theta = (\mathbf{X}^{\top}\mathbf{X})^{-1}\mathbf{X}^{\top}\mathbf{y}$$

|  | Gradient Descent | Normal Equation |
|---|---|---|
| Complexity | O(k·n·p) — k = iterations | O(p³) — inverting a p×p matrix |
| Best for | Large p (many features), works for any differentiable loss | Small p (< ~10,000 features), linear regression only |
| Requires feature scaling? | Yes — for reasonable convergence speed | No |
| Requires learning rate α? | Yes — must tune | No hyperparameters |
| Handles XᵀX singular (collinear features)? | Still converges (slowly) | Fails — inverse doesn't exist (fix: Ridge, see next-but-one chapter) |

## Animated Gradient Descent

Gradient descent iteratively moves θ in the direction that reduces J(θ). The update rule is:

$$\theta := \theta - \alpha\cdot\nabla J(\theta) \quad \text{where } \alpha = \text{learning rate}$$

Watch the algorithm find the optimal line through Surat textile export data (fabric metres vs ₹ revenue):

## The Normal Equation — Closed-Form Solution

For small datasets we can solve for θ analytically (no iterations needed):

$$\theta^{*} = (\mathbf{X}^{\top}\mathbf{X})^{-1}\mathbf{X}^{\top}\mathbf{y}$$

Drawback: inverting an (p+1)×(p+1) matrix costs O(p³) — impractical when p > ~10,000. Gradient descent scales better.

## From Scratch with NumPy

In [ ]:
# Linear Regression from scratch — Mehta Textiles Surat
import numpy as np

# Fabric produced (thousands of metres) vs Revenue (₹ lakhs)
X_raw = np.array([12,18,22,28,31,35,40,44,48,53,
                  58,62,67,72,78,83,88,94,100,106], dtype=float)
y     = np.array([14,21,26,33,36,42,48,52,57,62,
                  68,73,79,85,91,97,104,111,118,125], dtype=float)

# Add bias column
X = np.column_stack([np.ones(len(X_raw)), X_raw])   # shape (20, 2)
m = len(y)

# ── Gradient Descent ──────────────────────────────────────────
theta = np.zeros(2)
alpha, epochs = 0.0001, 10000

for ep in range(epochs):
    y_hat = X @ theta                           # matrix multiply
    grad  = X.T @ (y_hat - y) / m              # ∇J(θ)
    theta -= alpha * grad

print(f"θ₀ (intercept) = {theta[0]:.2f}")
print(f"θ₁ (slope)     = {theta[1]:.4f}")
print(f"Predict 120k m → ₹{theta[0]+theta[1]*120:.1f} lakhs")

# ── Normal Equation ───────────────────────────────────────────
theta_ne = np.linalg.inv(X.T @ X) @ X.T @ y
print(f"\nNormal Eq → θ₀={theta_ne[0]:.2f}, θ₁={theta_ne[1]:.4f}")

# ── Metrics ───────────────────────────────────────────────────
y_pred = X @ theta
mse  = np.mean((y_pred - y)**2)
rmse = np.sqrt(mse)
ss_res = np.sum((y - y_pred)**2)
ss_tot = np.sum((y - y.mean())**2)
r2   = 1 - ss_res/ss_tot
print(f"\nMSE={mse:.2f}  RMSE={rmse:.2f}  R²={r2:.4f}")

## With scikit-learn

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

X_feat = X_raw.reshape(-1, 1)
X_train, X_test, y_train, y_test = train_test_split(
    X_feat, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Intercept : {model.intercept_:.2f}")
print(f"Coefficient: {model.coef_[0]:.4f}")
print(f"Test RMSE  : {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
print(f"Test R²    : {r2_score(y_test, y_pred):.4f}")

## Reading an OLS Summary Table (statsmodels)

`LinearRegression` from scikit-learn hands you coefficients and predictions but stays silent on *how confident* you should be in them. **statsmodels**' `OLS` class fits the identical line but also runs the full classical-statistics machinery behind it — standard errors, t-tests, an F-test — and prints it all in one readable table. This is the table you'll see quoted in almost every regression-based research paper or business report.

Mehta Textiles wants to explain monthly revenue using two drivers — advertising spend and store footfall:

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

np.random.seed(42)
n = 60
ad_spend = np.random.uniform(2, 20, n)          # ₹ lakh / month
footfall = np.random.uniform(500, 5000, n)      # customers / month
revenue  = 8 + 1.35*ad_spend + 0.006*footfall + np.random.normal(0, 3, n)

df = pd.DataFrame({'ad_spend_lakh': ad_spend, 'footfall': footfall, 'revenue_lakh': revenue})
X = sm.add_constant(df[['ad_spend_lakh', 'footfall']])
model = sm.OLS(df['revenue_lakh'], X).fit()
print(model.summary())

Reading this table line by line:

| Row | Meaning |
|---|---|
| **R-squared** | 93.7% of the variance in revenue is explained by ad spend + footfall together. |
| **Adj. R-squared** | R² adjusted for the number of predictors (2 here) — penalises adding useless features. Always ≤ R²; a big gap between the two warns of overfitting with too many predictors for too little data. |
| **F-statistic** & **Prob (F-statistic)** | Tests the joint null hypothesis "all coefficients are zero." F=425.0 with Prob≈5.63e-35 (essentially 0) says the model as a whole is overwhelmingly significant — not a fluke. |
| **Log-Likelihood** | How probable the observed data is under the fitted model (higher = better fit). Only meaningful when *comparing* models on the same data — not interpretable on its own. |
| **AIC** / **BIC** | Log-Likelihood converted into a model-comparison score that penalises extra parameters — lower is better. BIC penalises complexity more harshly than AIC. Used to pick between competing models (e.g. 2-predictor vs. 5-predictor), never as a stand-alone number. |
| **coef / std err / t / P>\|t\|** | Per-feature version of the same idea: footfall's coefficient (0.0063) is 22.4 standard errors from zero (t=22.421), so P>\|t\| rounds to 0.000 — footfall is a statistically significant driver of revenue on its own, holding ad spend fixed. |
| **Cond. No.** | A multicollinearity smoke-alarm — large values (here 8.96e+03) suggest checking predictor correlations. The dedicated diagnostic for this, **Variance Inflation Factor (VIF)**, is covered in depth on the Multiple Regression page. |

> **🔗 sklearn vs. statsmodels**
>
> Use **scikit-learn** when the goal is prediction and the model feeds into a pipeline. Use **statsmodels** when the goal is inference — explaining *which* features matter and how confident you are — which is why business and research reports almost always quote a statsmodels-style table rather than raw sklearn coefficients.

## Key Assumptions (LINE)

| Assumption | What it means | How to check |
|---|---|---|
| **L**inearity | y is a linear function of X | Scatter plot, residuals vs fitted |
| **I**ndependence | Residuals are uncorrelated | Durbin-Watson test |
| **N**ormality | Residuals ~ N(0, σ²) | QQ-plot, Shapiro-Wilk |
| **E**qual variance | Homoscedasticity | Breusch-Pagan test |

## Regularisation

| Technique | Penalty added to J(θ) | Effect | Use when |
|---|---|---|---|
| **Ridge (L2)** | λ Σ θⱼ² | Shrinks all weights | Multicollinearity |
| **Lasso (L1)** | λ Σ \|θⱼ\| | Zeros out irrelevant features | Feature selection |
| **ElasticNet** | λ₁ L1 + λ₂ L2 | Balance of both | Many correlated features |

## When to Use / Avoid

### ✓ Use Linear Regression when

- Relationship is genuinely linear
- Interpretability is critical (banking, regulatory)
- Baseline model before complex algorithms
- Small dataset with few features
- HDFC Bank loan amount prediction

### ✗ Avoid when

- Non-linear patterns exist
- Outliers are heavy (use Huber regression)
- Features are strongly multicollinear (use Ridge)
- Target is categorical (use logistic regression)
- Complex interactions between features

## A Robust Alternative: Least Absolute Deviation (LAD) Regression

Ordinary Least Squares minimises *squared* error — which means a single extreme outlier gets squared too, and can pull the whole fitted line toward it disproportionately. Least Absolute Deviation regression swaps the loss function from squared to absolute error:

$$J_{\text{OLS}}(\theta) = \frac{1}{2m}\sum_i(\hat{y}_i-y_i)^2 \quad\longrightarrow\quad J_{\text{LAD}}(\theta) = \frac{1}{m}\sum_i|\hat{y}_i-y_i|$$

This single change in the loss function — nothing else about the model — makes the fit dramatically more robust to outliers, since an outlier's contribution to the loss grows linearly instead of quadratically with its error. The trade-off: LAD's loss isn't differentiable at zero error, so it can't be solved with the closed-form Normal Equation and needs an iterative solver (e.g., `statsmodels`' quantile regression at the median, which is mathematically equivalent to LAD).

In [ ]:
import statsmodels.formula.api as smf
import pandas as pd
import numpy as np

# Ahmedabad apartment prices with a few data-entry-error outliers
np.random.seed(2)
area = np.random.randint(500,2500,200)
price = 0.08*area + np.random.normal(0,5,200) + 40
price[5], price[40] = 900, 850  # two badly mis-entered rows
df = pd.DataFrame({'area':area, 'price':price})

ols = smf.ols('price ~ area', data=df).fit()
lad = smf.quantreg('price ~ area', data=df).fit(q=0.5)  # median regression = LAD

print(f"OLS slope: {ols.params['area']:.4f}   (distorted by the 2 outliers)")
print(f"LAD slope: {lad.params['area']:.4f}   (much closer to the true ₹0.08/sqft)")

LAD is a special case of **quantile regression** at the 50th percentile (the median) — a useful thing to know, since it means the same machinery generalises to predicting any quantile (e.g., the 90th percentile for conservative capacity planning), not just the central tendency.

> **🔗 Real-World Link — Salary Prediction**
>
> A 30-row dataset of years-of-experience vs. salary fits a single-feature line with R²=0.957 — a rare real dataset clean enough to show the textbook case with almost no noise. [See the case study →](https://statso.io/salary-prediction-case-study/) ·

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · The normal equation

`y` is exactly `3 + 2x`. Add a bias column to `x`, solve θ = (XᵀX)⁻¹Xᵀy with NumPy and store it in `theta` (expect `[3, 2]`).

In [ ]:
import numpy as np
x = np.arange(1, 9, dtype=float)
y = 3 + 2 * x
theta = None   # TODO


In [ ]:
try:
    check("intercept and slope", np.allclose(theta, [3, 2]))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
x = np.arange(1, 9, dtype=float)
y = 3 + 2 * x
X = np.column_stack([np.ones(len(x)), x])
theta = np.linalg.solve(X.T @ X, X.T @ y)

```

</details>

### Exercise 2 · Medium · Fit with scikit-learn and score it

Fit a `LinearRegression` on the training split, then store the slope in `slope`, the test R² in `r2` and the prediction for 60k metres in `pred_60`.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
rng = np.random.default_rng(0)
fabric = rng.uniform(10, 110, 80)
revenue = 2 + 1.2 * fabric + rng.normal(0, 2, 80)
X_tr, X_te, y_tr, y_te = train_test_split(fabric.reshape(-1, 1), revenue, test_size=0.25, random_state=1)
slope = r2 = pred_60 = None   # TODO


In [ ]:
try:
    check("slope near 1.2", abs(slope - 1.2) < 0.05)
    check("R2 is high", r2 > 0.99)
    check("prediction near 74", abs(pred_60 - 74) < 2)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
rng = np.random.default_rng(0)
fabric = rng.uniform(10, 110, 80)
revenue = 2 + 1.2 * fabric + rng.normal(0, 2, 80)
X_tr, X_te, y_tr, y_te = train_test_split(fabric.reshape(-1, 1), revenue, test_size=0.25, random_state=1)
model = LinearRegression().fit(X_tr, y_tr)
slope = model.coef_[0]
r2 = r2_score(y_te, model.predict(X_te))
pred_60 = model.predict([[60]])[0]

```

</details>

### Exercise 3 · Stretch · Gradient descent from scratch

Write `gd(x, y, lr, epochs)` that fits `y = θ0 + θ1·x` by batch gradient descent on the mean squared error (start at zeros) and returns `[θ0, θ1]`.

In [ ]:
import numpy as np
def gd(x, y, lr, epochs):
    pass   # TODO


In [ ]:
try:
    x = np.linspace(0, 1, 20)
    y = 1 + 2 * x
    theta = gd(x, y, lr=0.5, epochs=3000)
    check("recovers [1, 2]", np.allclose(theta, [1, 2], atol=1e-2))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
def gd(x, y, lr, epochs):
    t0 = t1 = 0.0
    m = len(x)
    for _ in range(epochs):
        err = t0 + t1 * x - y
        t0 -= lr * err.sum() / m
        t1 -= lr * (err * x).sum() / m
    return np.array([t0, t1])

```

The two gradients are the average error and the average error-times-x; each epoch steps both downhill.

</details>

---
*Back to the course: **Machine Learning End To End → Linear Regression**.*